# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(url)
# Retrieve metadata as a dict for display purposes
metadata_dict = dataset.metadata.to_json()
print(f"Dataset name: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")

## 2. Data Overview
List available record sets and their `@id`, fields, and columns by `@id`. 

We use Croissant schema's record set, field, and column `@id` for all references, as recommended.

In [ ]:
# Display record sets, fields, and their `@id`s
print('Available record sets:')
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- Record set name: {rs.name}, @id: {rs.id}")
    # Get fields for each record set
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"    - Field: {getattr(f, 'name', 'N/A')} (@id: {getattr(f, 'id', 'N/A')})")
            # If field refers to a column
            if hasattr(f, 'column') and f.column is not None:
                column = f.column
                col_name = getattr(column, 'name', getattr(column, 'id', 'N/A'))
                print(f"        - Column: {col_name} (@id: {getattr(column, 'id', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract all record sets by their @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set {rs_id}")

# If there are record sets, show the first one
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"DataFrame columns for record set {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data to prepare for further analysis.

The example below demonstrates EDA on the first record set, referencing all columns by their Croissant `@id`. You can adapt to explore specific fields or apply domain-specific filtering.

In [ ]:
import numpy as np

# Choose the first record set for demonstration
if record_set_ids:
    record_set_id = example_record_set_id
    df = dataframes[record_set_id]
    
    # Display numeric columns by @id
    numeric_fields = []
    for f in dataset.metadata.record_sets[0].fields:
        # Inspect field's data type
        data_type = getattr(f, 'data_type', '').lower() if hasattr(f, 'data_type') else ''
        if data_type in ['number', 'float', 'integer']:
            numeric_fields.append(f.id)
    print(f"Numeric field @ids: {numeric_fields}")

    # Pick a numeric field to analyze
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing field {numeric_field}")
        # Remove missing or non-numeric
        df = df.copy()
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered rows with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows.")
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print("Sample after normalization:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Attempt to group by a string field if present
        string_fields = [f.id for f in dataset.metadata.record_sets[0].fields if getattr(f, 'data_type', '').lower() == 'text']
        group_field = string_fields[0] if string_fields else None
        
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable string field for grouping found.")
    else:
        print("No numeric fields found to analyze.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between numeric fields in the dataset. This example uses the first available numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    # Plot the distribution of the first numeric field
    fig, ax = plt.subplots(figsize=(7,4))
    
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field} in {record_set_id}")
    ax.set_xlabel(numeric_field)
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset—'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells'—using the `mlcroissant` library. We demonstrated iterating over the Croissant schema's record sets and fields, performed EDA by normalizing and filtering data using field `@id`s, and visualized numeric fields. 

All processing referenced Croissant `@id` identifiers for transparency and reproducibility. 

Further analyses can refine criteria (e.g., select specific proteins or peptides), perform cross-record-set joins, or examine experimental metadata, always referencing schema entities by their `@id`.